# Parking Occupancy Prediction

The goal of this project is to predict parking lot occupancy
using historical data from Lisbon parking facilities.

## Workflow
1. Load and clean the data.
2. Select parking lots with sufficient historical coverage.
3. Extract geographic and calendar features.
4. Split the data chronologically.
5. Train a baseline Random Forest model.
6. Evaluate the model against a simple baseline.
7. Save the trained model and metadata.

Next steps: Optuna hyperparameter optimization, MLflow experiment
tracking, Docker, FastAPI, and AWS deployment.

In [ ]:
import ast
import os
import joblib
import numpy as np
import pandas as pd
import optuna
import mlflow
import mlflow.sklearn

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)

pd.set_option("display.max_columns", None)

## 1. Load and combine datasets

We load three CSV files containing historical parking occupancy.
The datetime column uses mixed date formats, so we parse it
with pandas and remove rows with missing timestamps or targets.

In [ ]:
files = [
    "Lisbon_parking1.csv",
    "Lisbon_parking2.csv",
    "Lisbon_parking4.csv",
]

frames = []

for filename in files:
    part = pd.read_csv(filename, sep=";")

    part["source_file"] = filename
    part["timestamp"] = pd.to_datetime(
        part["datetime"],
        format="mixed",
        dayfirst=True,
        errors="coerce",
    )

    frames.append(part)

df = pd.concat(frames, ignore_index=True)

df = df.dropna(
    subset=["timestamp", "occupancy", "max_capacity"]
).copy()

print("Rows after loading:", len(df))
display(df.head())

In [ ]:
file_counts = df.groupby("id_park")["source_file"].nunique()

selected_ids = file_counts[file_counts == 3].index

df = df[df["id_park"].isin(selected_ids)].copy()

print("Number of selected parking lots:", len(selected_ids))
print("Selected IDs:", sorted(selected_ids))
print("Rows after filtering:", len(df))

In [ ]:
rows_before = len(df)

df = df.drop_duplicates(
    subset=["id_park", "timestamp", "occupancy"]
).copy()

print("Rows before deduplication:", rows_before)
print("Rows after deduplication:", len(df))
print("Duplicates removed:", rows_before - len(df))

In [ ]:
def extract_coordinates(value):
    try:
        coordinates = ast.literal_eval(value)["coordinates"]
        return coordinates[0], coordinates[1]
    except (ValueError, SyntaxError, TypeError, KeyError):
        return np.nan, np.nan


df[["lon", "lat"]] = df["position"].apply(
    lambda value: pd.Series(extract_coordinates(value))
)

rows_before = len(df)
df = df.dropna(subset=["lon", "lat"]).copy()

print("Rows removed due to invalid coordinates:", rows_before - len(df))
display(df[["id_park", "position", "lon", "lat"]].head())

In [ ]:
df["hour"] = df["timestamp"].dt.hour
df["day_of_week"] = df["timestamp"].dt.dayofweek
df["month"] = df["timestamp"].dt.month
df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)

display(
    df[
        ["timestamp", "hour", "day_of_week", "month", "is_weekend"]
    ].head()
)

In [ ]:
FEATURES = [
    "max_capacity",
    "hour",
    "day_of_week",
    "month",
    "is_weekend",
    "lon",
    "lat",
]

TARGET = "occupancy"

print("Features:", FEATURES)
print("Target:", TARGET)

display(df[FEATURES + [TARGET]].head())

In [ ]:
timestamps = np.sort(df["timestamp"].unique())

train_index = int(len(timestamps) * 0.70)
val_index = int(len(timestamps) * 0.85)

train_end = timestamps[train_index]
val_end = timestamps[val_index]

train = df[df["timestamp"] < train_end].copy()

validation = df[
    (df["timestamp"] >= train_end)
    & (df["timestamp"] < val_end)
].copy()

test = df[df["timestamp"] >= val_end].copy()

X_train = train[FEATURES]
y_train = train[TARGET]

X_val = validation[FEATURES]
y_val = validation[TARGET]

X_test = test[FEATURES]
y_test = test[TARGET]

print("Train:", len(train), train["timestamp"].min(), "->", train["timestamp"].max())
print("Validation:", len(validation), validation["timestamp"].min(), "->", validation["timestamp"].max())
print("Test:", len(test), test["timestamp"].min(), "->", test["timestamp"].max())

In [ ]:
mlflow.set_tracking_uri("file:./mlruns")
mlflow.set_experiment("parking-occupancy-prediction")

print("MLflow tracking URI:", mlflow.get_tracking_uri())

In [ ]:
def objective(trial):
    params = {
        "n_estimators": trial.suggest_int(
            "n_estimators", 100, 300, step=100
        ),
        "max_depth": trial.suggest_categorical(
            "max_depth", [10, 15, 20, None]
        ),
        "min_samples_split": trial.suggest_int(
            "min_samples_split", 2, 20
        ),
        "min_samples_leaf": trial.suggest_int(
            "min_samples_leaf", 1, 8
        ),
        "max_features": trial.suggest_float(
            "max_features", 0.6, 1.0
        ),
    }

    with mlflow.start_run(
        run_name=f"optuna_trial_{trial.number}",
        nested=True,
    ):
        mlflow.log_params(params)

        model = RandomForestRegressor(
            **params,
            random_state=42,
            n_jobs=-1,
        )

        model.fit(X_train, y_train)

        val_predictions = model.predict(X_val)

        val_mae = mean_absolute_error(
            y_val,
            val_predictions,
        )

        mlflow.log_metric("validation_mae", val_mae)

        return val_mae

In [ ]:
study = optuna.create_study(
    direction="minimize",
    study_name="random-forest-optimization",
)

with mlflow.start_run(run_name="optuna_search"):
    study.optimize(
        objective,
        n_trials=10,
        show_progress_bar=True,
    )

print("Best validation MAE:", round(study.best_value, 2))
print("Best parameters:")
print(study.best_params)

In [ ]:
trials_df = study.trials_dataframe()
display(
    trials_df.sort_values("value")[
        ["number", "value", "params_n_estimators",
         "params_max_depth", "params_min_samples_split",
         "params_min_samples_leaf", "params_max_features"]
    ]
)

In [ ]:
train_full = pd.concat([train, validation]).sort_values("timestamp")

X_train_full = train_full[FEATURES]
y_train_full = train_full[TARGET]

best_model = RandomForestRegressor(
    **study.best_params,
    random_state=42,
    n_jobs=-1,
)

best_model.fit(X_train_full, y_train_full)

test_predictions = best_model.predict(X_test)

test_mae = mean_absolute_error(y_test, test_predictions)
test_rmse = np.sqrt(mean_squared_error(y_test, test_predictions))
test_r2 = r2_score(y_test, test_predictions)

print("=== FINAL TEST RESULTS ===")
print("MAE:", round(test_mae, 2))
print("RMSE:", round(test_rmse, 2))
print("R²:", round(test_r2, 4))

In [ ]:
os.makedirs("model", exist_ok=True)

metadata = {
    "features": FEATURES,
    "target": TARGET,
    "model_type": "RandomForestRegressor",
    "best_params": study.best_params,
    "validation_mae": float(study.best_value),
    "test_mae": float(test_mae),
    "test_rmse": float(test_rmse),
    "test_r2": float(test_r2),
}

joblib.dump(best_model, "model/parking-model.joblib")
joblib.dump(metadata, "model/metadata.joblib")

with mlflow.start_run(run_name="final_random_forest"):
    mlflow.log_params(study.best_params)

    mlflow.log_metrics({
        "best_validation_mae": float(study.best_value),
        "test_mae": float(test_mae),
        "test_rmse": float(test_rmse),
        "test_r2": float(test_r2),
    })

    mlflow.sklearn.log_model(
        sk_model=best_model,
        artifact_path="model",
    )

    mlflow.log_artifact("model/metadata.joblib")

print("Saved model and metadata.")

In [ ]:
# Calculate each parking lot's mean occupancy using training data only
lot_means = train.groupby("id_park")["occupancy"].mean()

# Predict test occupancy using the training-period mean
baseline_predictions = (
    test["id_park"]
    .map(lot_means)
    .fillna(train["occupancy"].mean())
)

# Evaluate baseline
baseline_mae = mean_absolute_error(
    y_test, baseline_predictions
)
baseline_rmse = np.sqrt(
    mean_squared_error(y_test, baseline_predictions)
)
baseline_r2 = r2_score(y_test, baseline_predictions)

print("=== BASELINE: PARKING LOT MEAN ===")
print("MAE:", round(baseline_mae, 2))
print("RMSE:", round(baseline_rmse, 2))
print("R²:", round(baseline_r2, 4))

# Compare against the final Optuna-tuned model
print("\n=== OPTUNA MODEL vs BASELINE ===")
print("Optuna model MAE:", round(test_mae, 2))
print("Baseline MAE:", round(baseline_mae, 2))

if baseline_mae > 0:
    improvement = (
        (baseline_mae - test_mae) / baseline_mae * 100
    )
    print("MAE improvement:", round(improvement, 2), "%")

## Results and next steps

The baseline Random Forest model has been trained and evaluated
against a parking-lot mean baseline.

Next steps:
1. Introduce a chronological validation split.
2. Use Optuna to optimize hyperparameters.
3. Track experiments and metrics with MLflow.
4. Evaluate the selected model once on the held-out test period.
5. Integrate the model with FastAPI and Docker.
6. Deploy the application and MLflow tracking server on AWS.